# Cross-Validation Strategies

**Interview Focus**: K-Fold vs stratified, time-series splits, nested CV.

**Key Concepts**:
- K-Fold: split data into K folds, train on K-1, validate on 1
- Stratified K-Fold: preserves class distribution in each fold
- Time-Series Split: respects temporal order (no look-ahead bias)
- Nested CV: unbiased model selection + performance estimation

**Math Notes**:
- CV estimate: $\hat{E} = \frac{1}{K} \sum_{k=1}^{K} L(f_{-k}, D_k)$ where $f_{-k}$ is trained without fold k
- Bias-variance of CV: K large -> low bias, high variance (leave-one-out extreme)
- Nested CV: outer loop estimates generalization, inner loop selects hyperparameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. K-Fold Cross-Validation from Scratch

In [ ]:
def k_fold_split(n_samples, k=5, shuffle=True, seed=42):
    """Generate K-Fold train/val index splits.
    
    Yields (train_indices, val_indices) for each fold.
    """
    indices = np.arange(n_samples)
    if shuffle:
        rng = np.random.RandomState(seed)
        rng.shuffle(indices)
    
    fold_sizes = np.full(k, n_samples // k, dtype=int)
    fold_sizes[:n_samples % k] += 1  # distribute remainder
    
    current = 0
    folds = []
    for fold_size in fold_sizes:
        val_indices = indices[current:current + fold_size]
        train_indices = np.concatenate([indices[:current], indices[current + fold_size:]])
        folds.append((train_indices, val_indices))
        current += fold_size
    
    return folds


# Demo
n = 20
k = 5
folds = k_fold_split(n, k=k, shuffle=True)

print(f"{k}-Fold CV on {n} samples:\n")
for i, (train_idx, val_idx) in enumerate(folds):
    print(f"Fold {i+1}: Train ({len(train_idx)} samples): {sorted(train_idx)}")
    print(f"         Val   ({len(val_idx)} samples): {sorted(val_idx)}")
    print()

# Verify: every sample appears in exactly one validation fold
all_val = np.concatenate([val for _, val in folds])
print(f"Every sample in exactly one val fold: {len(np.unique(all_val)) == n}")

In [ ]:
def visualize_cv_splits(folds, n_samples, title='Cross-Validation Splits', labels=None):
    """Visualize train/val splits as a grid."""
    n_folds = len(folds)
    fig, ax = plt.subplots(figsize=(12, 0.6 * n_folds + 1.5))
    
    cmap = {'train': '#2196F3', 'val': '#F44336', 'unused': '#E0E0E0'}
    
    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        # Create color array
        colors = np.full(n_samples, 2)  # unused
        colors[train_idx] = 0  # train
        colors[val_idx] = 1    # val
        
        for i in range(n_samples):
            if colors[i] == 0:
                c = cmap['train']
            elif colors[i] == 1:
                c = cmap['val']
            else:
                c = cmap['unused']
            ax.barh(fold_idx, 1, left=i, height=0.8, color=c, edgecolor='white', linewidth=0.5)
        
        if labels is not None and fold_idx < len(labels):
            ax.text(-0.5, fold_idx, labels[fold_idx], ha='right', va='center', fontsize=9)
    
    ax.set_yticks(range(n_folds))
    ax.set_yticklabels([f'Fold {i+1}' for i in range(n_folds)] if labels is None else [''] * n_folds)
    ax.set_xlabel('Sample Index')
    ax.set_title(title)
    ax.invert_yaxis()
    
    # Legend
    train_patch = mpatches.Patch(color=cmap['train'], label='Train')
    val_patch = mpatches.Patch(color=cmap['val'], label='Validation')
    ax.legend(handles=[train_patch, val_patch], loc='upper right')
    
    plt.tight_layout()
    return fig


# Visualize K-Fold
n = 50
folds_5 = k_fold_split(n, k=5, shuffle=True)
fig = visualize_cv_splits(folds_5, n, '5-Fold Cross-Validation')
plt.show()

## 2. K-Fold CV for Model Evaluation

In [ ]:
def simple_logistic_regression(X_train, y_train, X_val, lr=0.1, n_iter=100):
    """Minimal logistic regression for demo purposes."""
    n_features = X_train.shape[1]
    w = np.zeros(n_features)
    b = 0.0
    
    for _ in range(n_iter):
        z = X_train @ w + b
        p = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
        
        # Gradients
        error = p - y_train
        grad_w = X_train.T @ error / len(y_train)
        grad_b = np.mean(error)
        
        w -= lr * grad_w
        b -= lr * grad_b
    
    # Predict on validation
    z_val = X_val @ w + b
    p_val = 1 / (1 + np.exp(-np.clip(z_val, -500, 500)))
    y_pred = (p_val > 0.5).astype(int)
    return y_pred


def cross_validate(X, y, k=5, seed=42):
    """Run K-Fold cross-validation and return per-fold accuracies."""
    folds = k_fold_split(len(y), k=k, shuffle=True, seed=seed)
    fold_accuracies = []
    
    for train_idx, val_idx in folds:
        X_train, y_train = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        
        y_pred = simple_logistic_regression(X_train, y_train, X_val)
        acc = np.mean(y_pred == y_val)
        fold_accuracies.append(acc)
    
    return fold_accuracies


# Generate synthetic data
np.random.seed(42)
n_samples = 300
n_features = 5

X = np.random.randn(n_samples, n_features)
true_w = np.array([1.5, -1.0, 0.5, 0.0, 0.0])
logits = X @ true_w + 0.3 * np.random.randn(n_samples)
y = (logits > 0).astype(int)

# Run CV
fold_accs = cross_validate(X, y, k=5)

print(f"5-Fold CV Results:")
for i, acc in enumerate(fold_accs):
    print(f"  Fold {i+1}: {acc:.4f}")
print(f"\n  Mean accuracy: {np.mean(fold_accs):.4f} +/- {np.std(fold_accs):.4f}")

## 3. Stratified K-Fold

Standard K-Fold can create folds with very different class distributions, especially with imbalanced data. Stratified K-Fold ensures each fold has approximately the same class proportions.

In [ ]:
def stratified_k_fold_split(y, k=5, shuffle=True, seed=42):
    """Stratified K-Fold: preserves class proportions in each fold."""
    rng = np.random.RandomState(seed)
    n = len(y)
    classes = np.unique(y)
    
    # Separate indices by class
    class_indices = {}
    for cls in classes:
        idx = np.where(y == cls)[0]
        if shuffle:
            rng.shuffle(idx)
        class_indices[cls] = idx
    
    # Assign each class's indices to folds in round-robin
    fold_indices = [[] for _ in range(k)]
    for cls in classes:
        idx = class_indices[cls]
        fold_sizes = np.full(k, len(idx) // k, dtype=int)
        fold_sizes[:len(idx) % k] += 1
        
        current = 0
        for fold_idx, size in enumerate(fold_sizes):
            fold_indices[fold_idx].extend(idx[current:current + size])
            current += size
    
    # Convert to train/val splits
    folds = []
    for fold_idx in range(k):
        val_idx = np.array(fold_indices[fold_idx])
        train_idx = np.concatenate([fold_indices[j] for j in range(k) if j != fold_idx])
        folds.append((train_idx, val_idx))
    
    return folds


# Demo: imbalanced data
np.random.seed(42)
n_total = 100
y_imbalanced = np.array([1]*10 + [0]*90)  # 10% positive

# Compare standard vs stratified K-Fold
folds_standard = k_fold_split(n_total, k=5)
folds_stratified = stratified_k_fold_split(y_imbalanced, k=5)

print("Standard K-Fold — class distribution per fold:")
for i, (train_idx, val_idx) in enumerate(folds_standard):
    pos_rate = y_imbalanced[val_idx].mean()
    print(f"  Fold {i+1}: {len(val_idx)} samples, positive rate = {pos_rate:.2%}")

print("\nStratified K-Fold — class distribution per fold:")
for i, (train_idx, val_idx) in enumerate(folds_stratified):
    pos_rate = y_imbalanced[val_idx].mean()
    print(f"  Fold {i+1}: {len(val_idx)} samples, positive rate = {pos_rate:.2%}")

print(f"\nOverall positive rate: {y_imbalanced.mean():.2%}")
print("Stratified keeps each fold close to 10%. Standard can vary wildly.")

In [ ]:
# Visualize standard vs stratified
fig, axes = plt.subplots(2, 1, figsize=(12, 5))

for ax, folds_list, title in [(axes[0], folds_standard, 'Standard K-Fold (may have uneven classes)'),
                               (axes[1], folds_stratified, 'Stratified K-Fold (balanced classes)')]:
    n_folds = len(folds_list)
    for fold_idx, (train_idx, val_idx) in enumerate(folds_list):
        for i in val_idx:
            color = '#F44336' if y_imbalanced[i] == 1 else '#2196F3'
            ax.barh(fold_idx, 1, left=i, height=0.8, color=color, edgecolor='white', linewidth=0.3)
        for i in train_idx:
            color = '#FFCDD2' if y_imbalanced[i] == 1 else '#BBDEFB'
            ax.barh(fold_idx, 1, left=i, height=0.8, color=color, edgecolor='white', linewidth=0.3)
    
    ax.set_yticks(range(n_folds))
    ax.set_yticklabels([f'Fold {i+1}' for i in range(n_folds)])
    ax.set_title(title)
    ax.invert_yaxis()

# Legend
handles = [
    mpatches.Patch(color='#F44336', label='Val (pos)'),
    mpatches.Patch(color='#2196F3', label='Val (neg)'),
    mpatches.Patch(color='#FFCDD2', label='Train (pos)'),
    mpatches.Patch(color='#BBDEFB', label='Train (neg)'),
]
axes[0].legend(handles=handles, loc='upper right', fontsize=8)
axes[1].set_xlabel('Sample Index')

plt.tight_layout()
plt.show()

## 4. Impact on Evaluation Variance

In [ ]:
# Show that stratified CV has lower variance on imbalanced data
np.random.seed(42)
n = 200
n_features = 5

X_imb = np.random.randn(n, n_features)
# 10% positive rate
y_imb = np.zeros(n, dtype=int)
y_imb[:20] = 1

# Give positive class a signal
X_imb[:20, 0] += 2.0

# Run multiple rounds of CV with different seeds
n_rounds = 20
standard_accs = []
stratified_accs = []

for seed in range(n_rounds):
    # Standard K-Fold
    folds_s = k_fold_split(n, k=5, shuffle=True, seed=seed)
    accs = []
    for train_idx, val_idx in folds_s:
        y_pred = simple_logistic_regression(X_imb[train_idx], y_imb[train_idx], X_imb[val_idx])
        accs.append(np.mean(y_pred == y_imb[val_idx]))
    standard_accs.append(np.mean(accs))
    
    # Stratified K-Fold
    folds_st = stratified_k_fold_split(y_imb, k=5, shuffle=True, seed=seed)
    accs = []
    for train_idx, val_idx in folds_st:
        y_pred = simple_logistic_regression(X_imb[train_idx], y_imb[train_idx], X_imb[val_idx])
        accs.append(np.mean(y_pred == y_imb[val_idx]))
    stratified_accs.append(np.mean(accs))

fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot([standard_accs, stratified_accs], labels=['Standard K-Fold', 'Stratified K-Fold'])
ax.set_ylabel('Mean CV Accuracy')
ax.set_title(f'CV Accuracy Variance (Imbalanced Data, {n_rounds} rounds)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Standard K-Fold:   mean={np.mean(standard_accs):.4f}, std={np.std(standard_accs):.4f}")
print(f"Stratified K-Fold: mean={np.mean(stratified_accs):.4f}, std={np.std(stratified_accs):.4f}")
print("\nStratified CV gives more stable (lower variance) estimates.")

## 5. Time-Series Split

Standard K-Fold is **invalid** for time-series because:
- Training on future data to predict the past introduces **look-ahead bias**
- Temporal autocorrelation means shuffled folds are not independent

Two common strategies:
1. **Expanding window**: training set grows over time
2. **Sliding window**: fixed-size training window

In [ ]:
def time_series_split_expanding(n_samples, n_splits=5, min_train_size=None):
    """Expanding window time-series split.
    
    Each split uses all previous data for training.
    """
    if min_train_size is None:
        min_train_size = n_samples // (n_splits + 1)
    
    test_size = (n_samples - min_train_size) // n_splits
    
    folds = []
    for i in range(n_splits):
        train_end = min_train_size + i * test_size
        test_end = min(train_end + test_size, n_samples)
        
        train_idx = np.arange(0, train_end)
        val_idx = np.arange(train_end, test_end)
        
        if len(val_idx) > 0:
            folds.append((train_idx, val_idx))
    
    return folds


def time_series_split_sliding(n_samples, n_splits=5, train_size=None, test_size=None):
    """Sliding window time-series split.
    
    Fixed-size training window slides forward.
    """
    if train_size is None:
        train_size = n_samples // (n_splits + 1)
    if test_size is None:
        test_size = train_size // 3
    
    step = (n_samples - train_size - test_size) // max(n_splits - 1, 1)
    
    folds = []
    for i in range(n_splits):
        train_start = i * step
        train_end = train_start + train_size
        test_end = min(train_end + test_size, n_samples)
        
        if test_end <= n_samples:
            train_idx = np.arange(train_start, train_end)
            val_idx = np.arange(train_end, test_end)
            folds.append((train_idx, val_idx))
    
    return folds


# Demo
n = 100
folds_expanding = time_series_split_expanding(n, n_splits=5)
folds_sliding = time_series_split_sliding(n, n_splits=5, train_size=40, test_size=12)

print("Expanding Window:")
for i, (tr, val) in enumerate(folds_expanding):
    print(f"  Split {i+1}: Train [{tr[0]}-{tr[-1]}] ({len(tr)}), "
          f"Val [{val[0]}-{val[-1]}] ({len(val)})")

print("\nSliding Window:")
for i, (tr, val) in enumerate(folds_sliding):
    print(f"  Split {i+1}: Train [{tr[0]}-{tr[-1]}] ({len(tr)}), "
          f"Val [{val[0]}-{val[-1]}] ({len(val)})")

In [ ]:
# Visualize all three splitting strategies
n = 100

fig, axes = plt.subplots(3, 1, figsize=(14, 8))

# Standard K-Fold (wrong for time series)
folds_kf = k_fold_split(n, k=5, shuffle=True)
ax = axes[0]
for fold_idx, (train_idx, val_idx) in enumerate(folds_kf):
    for i in range(n):
        if i in val_idx:
            c = '#F44336'
        else:
            c = '#2196F3'
        ax.barh(fold_idx, 1, left=i, height=0.8, color=c, edgecolor='white', linewidth=0.3)
ax.set_yticks(range(5))
ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_title('Standard K-Fold (WRONG for time series — leaks future data)')
ax.invert_yaxis()

# Expanding window
folds_exp = time_series_split_expanding(n, n_splits=5)
ax = axes[1]
for fold_idx, (train_idx, val_idx) in enumerate(folds_exp):
    for i in range(n):
        if i in val_idx:
            c = '#F44336'
        elif i in train_idx:
            c = '#2196F3'
        else:
            c = '#E0E0E0'
        ax.barh(fold_idx, 1, left=i, height=0.8, color=c, edgecolor='white', linewidth=0.3)
ax.set_yticks(range(len(folds_exp)))
ax.set_yticklabels([f'Split {i+1}' for i in range(len(folds_exp))])
ax.set_title('Expanding Window (training set grows)')
ax.invert_yaxis()

# Sliding window
folds_sld = time_series_split_sliding(n, n_splits=5, train_size=40, test_size=10)
ax = axes[2]
for fold_idx, (train_idx, val_idx) in enumerate(folds_sld):
    for i in range(n):
        if i in val_idx:
            c = '#F44336'
        elif i in train_idx:
            c = '#2196F3'
        else:
            c = '#E0E0E0'
        ax.barh(fold_idx, 1, left=i, height=0.8, color=c, edgecolor='white', linewidth=0.3)
ax.set_yticks(range(len(folds_sld)))
ax.set_yticklabels([f'Split {i+1}' for i in range(len(folds_sld))])
ax.set_title('Sliding Window (fixed-size training window)')
ax.set_xlabel('Time (sample index)')
ax.invert_yaxis()

# Common legend
handles = [
    mpatches.Patch(color='#2196F3', label='Train'),
    mpatches.Patch(color='#F44336', label='Validation'),
    mpatches.Patch(color='#E0E0E0', label='Unused'),
]
axes[0].legend(handles=handles, loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## 6. Why Time Series Needs Special Splitting

In [ ]:
# Demonstrate look-ahead bias with a simple time-series prediction task
np.random.seed(42)
n_ts = 200

# Generate a time series with a trend + noise
t = np.arange(n_ts)
trend = 0.05 * t
seasonal = 3 * np.sin(2 * np.pi * t / 50)
noise = np.random.randn(n_ts) * 0.5
y_ts = trend + seasonal + noise

# Features: lagged values
lookback = 5
X_ts = np.column_stack([y_ts[lookback-i-1:n_ts-i-1] for i in range(lookback)])
y_ts_target = y_ts[lookback:]
n_effective = len(y_ts_target)

def simple_linear_regression(X_train, y_train, X_val):
    """OLS linear regression."""
    # Add bias
    X_train_b = np.column_stack([np.ones(len(X_train)), X_train])
    X_val_b = np.column_stack([np.ones(len(X_val)), X_val])
    # Normal equations
    w = np.linalg.lstsq(X_train_b, y_train, rcond=None)[0]
    return X_val_b @ w

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

# Standard K-Fold (wrong: leaks future data)
folds_wrong = k_fold_split(n_effective, k=5, shuffle=True)
mse_wrong = []
for train_idx, val_idx in folds_wrong:
    y_pred = simple_linear_regression(X_ts[train_idx], y_ts_target[train_idx], X_ts[val_idx])
    mse_wrong.append(mse(y_ts_target[val_idx], y_pred))

# Time-series split (correct)
folds_correct = time_series_split_expanding(n_effective, n_splits=5)
mse_correct = []
for train_idx, val_idx in folds_correct:
    y_pred = simple_linear_regression(X_ts[train_idx], y_ts_target[train_idx], X_ts[val_idx])
    mse_correct.append(mse(y_ts_target[val_idx], y_pred))

print(f"Time-series prediction task (n={n_effective}, {lookback} lagged features):")
print(f"\n  Standard K-Fold (WRONG):  Mean MSE = {np.mean(mse_wrong):.4f} +/- {np.std(mse_wrong):.4f}")
print(f"  Time-Series Split (RIGHT): Mean MSE = {np.mean(mse_correct):.4f} +/- {np.std(mse_correct):.4f}")
print(f"\n  Standard K-Fold gives optimistically low MSE because it trains on future data.")
print(f"  The true out-of-sample error is HIGHER — time-series split is more honest.")

In [ ]:
# Plot the time series and predictions
fig, axes = plt.subplots(2, 1, figsize=(13, 7))

# Show the time series
ax = axes[0]
ax.plot(t, y_ts, 'b-', alpha=0.7, linewidth=1, label='Time series')
ax.plot(t, trend, 'r--', alpha=0.5, label='Trend')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.set_title('Synthetic Time Series (trend + seasonal + noise)')
ax.legend()
ax.grid(True, alpha=0.3)

# Compare MSEs
ax = axes[1]
x = np.arange(5)
width = 0.35
ax.bar(x - width/2, mse_wrong, width, label='Standard K-Fold (biased)', color='#F44336', alpha=0.8)
ax.bar(x + width/2, mse_correct, width, label='Time-Series Split (correct)', color='#2196F3', alpha=0.8)
ax.set_xlabel('Fold')
ax.set_ylabel('MSE')
ax.set_title('Per-Fold MSE: Standard vs Time-Series Split')
ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i+1}' for i in range(5)])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. Nested Cross-Validation

**Problem**: if you use the same CV to both select hyperparameters and estimate performance, you get an optimistically biased estimate.

**Solution**: nested CV with two loops:
- **Outer loop** (K folds): estimates true generalization performance
- **Inner loop** (J folds within each outer training set): selects best hyperparameters

In [ ]:
def nested_cross_validation(X, y, param_grid, k_outer=5, k_inner=3, seed=42):
    """Nested CV: outer loop for evaluation, inner loop for hyperparameter selection.
    
    Args:
        param_grid: dict of parameter name -> list of values
                    Currently supports 'lr' and 'n_iter' for logistic regression.
    """
    outer_folds = k_fold_split(len(y), k=k_outer, shuffle=True, seed=seed)
    
    outer_scores = []
    best_params_per_fold = []
    
    for fold_idx, (outer_train_idx, outer_test_idx) in enumerate(outer_folds):
        X_outer_train = X[outer_train_idx]
        y_outer_train = y[outer_train_idx]
        X_outer_test = X[outer_test_idx]
        y_outer_test = y[outer_test_idx]
        
        # Inner loop: find best hyperparameters
        best_inner_score = -1
        best_params = None
        
        # Generate all parameter combinations
        param_names = list(param_grid.keys())
        param_values = list(param_grid.values())
        
        # Simple grid (for 2 params)
        for lr in param_grid.get('lr', [0.1]):
            for n_iter in param_grid.get('n_iter', [100]):
                inner_folds = k_fold_split(len(y_outer_train), k=k_inner,
                                           shuffle=True, seed=seed + fold_idx)
                inner_scores = []
                
                for inner_train_idx, inner_val_idx in inner_folds:
                    X_inner_train = X_outer_train[inner_train_idx]
                    y_inner_train = y_outer_train[inner_train_idx]
                    X_inner_val = X_outer_train[inner_val_idx]
                    y_inner_val = y_outer_train[inner_val_idx]
                    
                    y_pred = simple_logistic_regression(
                        X_inner_train, y_inner_train, X_inner_val,
                        lr=lr, n_iter=n_iter)
                    inner_scores.append(np.mean(y_pred == y_inner_val))
                
                mean_inner_score = np.mean(inner_scores)
                if mean_inner_score > best_inner_score:
                    best_inner_score = mean_inner_score
                    best_params = {'lr': lr, 'n_iter': n_iter}
        
        # Train with best params on full outer training set, evaluate on outer test
        y_pred_outer = simple_logistic_regression(
            X_outer_train, y_outer_train, X_outer_test,
            lr=best_params['lr'], n_iter=best_params['n_iter'])
        outer_score = np.mean(y_pred_outer == y_outer_test)
        
        outer_scores.append(outer_score)
        best_params_per_fold.append(best_params)
    
    return outer_scores, best_params_per_fold


# Run nested CV
param_grid = {
    'lr': [0.01, 0.05, 0.1, 0.5],
    'n_iter': [50, 100, 200],
}

np.random.seed(42)
n = 300
X_nest = np.random.randn(n, 5)
true_w = np.array([1.5, -1.0, 0.5, 0.0, 0.0])
y_nest = (X_nest @ true_w + 0.3 * np.random.randn(n) > 0).astype(int)

outer_scores, best_params = nested_cross_validation(
    X_nest, y_nest, param_grid, k_outer=5, k_inner=3)

print(f"Nested CV Results (5-outer x 3-inner):")
for i, (score, params) in enumerate(zip(outer_scores, best_params)):
    print(f"  Outer Fold {i+1}: Accuracy = {score:.4f}, Best params = {params}")
print(f"\n  Mean accuracy: {np.mean(outer_scores):.4f} +/- {np.std(outer_scores):.4f}")
print(f"  This is an unbiased estimate of generalization with hyperparameter tuning.")

In [ ]:
# Compare: nested CV vs non-nested (biased) CV
# Non-nested: use same CV for both selection and evaluation

def non_nested_cv(X, y, param_grid, k=5, seed=42):
    """Non-nested CV: use same folds for selection and evaluation (biased!)."""
    folds = k_fold_split(len(y), k=k, shuffle=True, seed=seed)
    
    best_mean_score = -1
    best_params = None
    best_fold_scores = None
    
    for lr in param_grid.get('lr', [0.1]):
        for n_iter in param_grid.get('n_iter', [100]):
            fold_scores = []
            for train_idx, val_idx in folds:
                y_pred = simple_logistic_regression(
                    X[train_idx], y[train_idx], X[val_idx],
                    lr=lr, n_iter=n_iter)
                fold_scores.append(np.mean(y_pred == y[val_idx]))
            
            mean_score = np.mean(fold_scores)
            if mean_score > best_mean_score:
                best_mean_score = mean_score
                best_params = {'lr': lr, 'n_iter': n_iter}
                best_fold_scores = fold_scores
    
    return best_fold_scores, best_params


non_nested_scores, non_nested_params = non_nested_cv(X_nest, y_nest, param_grid)

print(f"Nested CV:     {np.mean(outer_scores):.4f} +/- {np.std(outer_scores):.4f}")
print(f"Non-nested CV: {np.mean(non_nested_scores):.4f} +/- {np.std(non_nested_scores):.4f}")
print(f"\nNon-nested is optimistically biased (higher) because it reports the best")
print(f"CV score found during hyperparameter search — overfitting to the validation folds.")

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([0], [np.mean(non_nested_scores)], yerr=[np.std(non_nested_scores)],
       width=0.5, color='#F44336', alpha=0.8, capsize=8, label='Non-nested (biased)')
ax.bar([1], [np.mean(outer_scores)], yerr=[np.std(outer_scores)],
       width=0.5, color='#2196F3', alpha=0.8, capsize=8, label='Nested (unbiased)')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Non-Nested CV', 'Nested CV'])
ax.set_ylabel('Accuracy')
ax.set_title('Nested vs Non-Nested CV')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 8. Visualize Nested CV Structure

In [ ]:
# Visualize the nested CV splits for one outer fold
n = 60
outer_folds = k_fold_split(n, k=5, shuffle=False)  # no shuffle for clarity

# Pick outer fold 0
outer_train, outer_test = outer_folds[0]
inner_folds = k_fold_split(len(outer_train), k=3, shuffle=False)

fig, ax = plt.subplots(figsize=(14, 4))

# Row 0: outer split
for i in range(n):
    if i in outer_test:
        c = '#F44336'  # outer test
    else:
        c = '#2196F3'  # outer train
    ax.barh(0, 1, left=i, height=0.8, color=c, edgecolor='white', linewidth=0.3)

# Rows 1-3: inner splits within outer training data
for fold_idx, (inner_train_idx, inner_val_idx) in enumerate(inner_folds):
    for i in range(n):
        if i in outer_test:
            c = '#E0E0E0'  # held out (outer test)
        elif np.isin(np.where(np.isin(np.arange(n), outer_train))[0],
                     outer_train[inner_val_idx]).any() and i in outer_train[inner_val_idx]:
            c = '#FF9800'  # inner val
        elif i in outer_train[inner_train_idx]:
            c = '#4CAF50'  # inner train
        else:
            c = '#E0E0E0'
        ax.barh(fold_idx + 1, 1, left=i, height=0.8, color=c, edgecolor='white', linewidth=0.3)

ax.set_yticks(range(4))
ax.set_yticklabels(['Outer Split', 'Inner Fold 1', 'Inner Fold 2', 'Inner Fold 3'])
ax.set_xlabel('Sample Index')
ax.set_title('Nested CV: Outer Fold 1 (inner folds shown)')
ax.invert_yaxis()

handles = [
    mpatches.Patch(color='#2196F3', label='Outer Train'),
    mpatches.Patch(color='#F44336', label='Outer Test'),
    mpatches.Patch(color='#4CAF50', label='Inner Train'),
    mpatches.Patch(color='#FF9800', label='Inner Val'),
    mpatches.Patch(color='#E0E0E0', label='Unused'),
]
ax.legend(handles=handles, loc='upper right', fontsize=8, ncol=5)

plt.tight_layout()
plt.show()

## 9. Choosing K: Bias-Variance Tradeoff

In [ ]:
# How does K affect the CV estimate?
np.random.seed(42)
n = 200
X_bv = np.random.randn(n, 5)
y_bv = (X_bv @ np.array([1.5, -1.0, 0.5, 0.0, 0.0]) + 0.3 * np.random.randn(n) > 0).astype(int)

k_values = [2, 3, 5, 10, 20, 50]
n_repeats = 30

means = []
stds = []

for k in k_values:
    cv_means = []
    for repeat in range(n_repeats):
        folds = k_fold_split(n, k=k, shuffle=True, seed=repeat)
        accs = []
        for train_idx, val_idx in folds:
            y_pred = simple_logistic_regression(X_bv[train_idx], y_bv[train_idx], X_bv[val_idx])
            accs.append(np.mean(y_pred == y_bv[val_idx]))
        cv_means.append(np.mean(accs))
    means.append(np.mean(cv_means))
    stds.append(np.std(cv_means))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(k_values, means, 'o-', color='#2196F3', linewidth=2, label='Mean CV estimate')
ax.fill_between(k_values, np.array(means) - np.array(stds),
                np.array(means) + np.array(stds), alpha=0.2, color='#2196F3')
ax.set_xlabel('K (number of folds)')
ax.set_ylabel('CV Accuracy Estimate')
ax.set_title('CV Estimate vs K')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(k_values, stds, 'o-', color='#F44336', linewidth=2)
ax.set_xlabel('K (number of folds)')
ax.set_ylabel('Std of CV Estimate')
ax.set_title('Variance of CV Estimate vs K')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("K=2: high bias (only 50% of data for training), low variance.")
print("K=N (LOO): low bias (N-1 training samples), high variance (correlated folds).")
print("K=5 or K=10: commonly used as a good tradeoff.")

## Interview Questions & Answers

---

**Q: K-Fold vs Stratified K-Fold?**

- **Standard K-Fold**: randomly partitions data into K folds. For imbalanced data, some folds may have very different class distributions, leading to high variance in CV estimates.
- **Stratified K-Fold**: ensures each fold has approximately the same class proportions as the full dataset. Strongly recommended for classification, especially with imbalanced classes.
- Always use stratified for classification. Standard K-Fold is fine for regression.

---

**Q: Nested CV — why?**

If you use the same CV for hyperparameter tuning AND performance estimation, the reported performance is optimistically biased (you picked the hyperparameters that looked best on these exact validation folds).

Nested CV fixes this:
- Outer loop (K folds): gives an unbiased performance estimate
- Inner loop (J folds): selects hyperparameters on each outer training set

Cost: K * J * |param_grid| model fits. Typical: 5-outer x 3-inner.

---

**Q: Cross-validation with time series?**

Standard K-Fold is **invalid** because it trains on future data (look-ahead bias). Use:
1. **Expanding window**: training set grows, always predicts future. Pros: uses all available data. Cons: different training set sizes across splits.
2. **Sliding window**: fixed-size training window. Pros: consistent training size. Cons: discards old data.

Key rule: validation data must always be AFTER training data in time.

---

**Q: Leave-One-Out CV vs K-Fold?**

LOO (K=N): nearly unbiased but extremely high variance (folds differ by only 1 sample, so they are highly correlated). Also computationally expensive. K=5 or K=10 usually give better bias-variance tradeoff and are the standard choice.

## Quick Reference

```
K-Fold:       Random split into K folds, rotate validation. O(K * train_cost)
Stratified:   Same, but preserves class distribution per fold
Time-Series:  Expanding or sliding window, always train on past
Nested CV:    Outer (evaluation) x Inner (tuning) — unbiased with HP selection

K=5 or K=10 is the standard choice (bias-variance tradeoff).

| Strategy     | When to Use                           |
|-------------|---------------------------------------|
| K-Fold      | Default for regression                |
| Stratified  | Classification (especially imbalanced)|
| Time-Series | Temporal/sequential data              |
| Nested CV   | When tuning hyperparameters           |
| Repeated CV | When you need tighter CI on CV estimate|
```